# 2026-09-06_1 Figure3：Cell type → Module pathway-source network

在已完成的 DEG-anchored pathway → module 框架上加**细胞类型维度**：
每个 fate 的功能 module 由哪些 cell types 的 pathways 构成？

- 单位：`direction × fate × celltype × module`；主指标 `n_unique_pathways`，配套 `n_anchor_genes` + list；
- 每个 fate 一张 **cell type → module alluvial**（左 celltype、右 module，连线宽=来源强度）；
- **连线宽两版**：pathway 数 与 anchor gene 数（计划 6.2：pathway 数可能虚高，需 anchor gene 对照）；
- **Macular 与 Peripheral 相同口径**，module/celltype 顺序一致，便于对比；
- celltype 顺序 `Rod→Cone→AC→BC→HC→MG`；不重跑 GSEA、不重定义 fate。

解释边界：跨 celltype 出现只说 module-level convergence，不说 shared genes/pathways；不跨 celltype 平均 NES。

In [ ]:
suppressPackageStartupMessages({library(dplyr); library(tidyr); library(readr); library(stringr)
  library(ggplot2); library(ggalluvial); library(patchwork)})

dir_d <- "/project/work/2026-09-01_GSEA_test/Data/"
dir_p <- "/project/work/2026-09-01_GSEA_test/Plot/"
cols_fate <- c(DM='#3C5488', DAL='#E64B35', AAM='#00A087')
# celltype 配色沿用 color.ipynb 的 cols.celltype（Paired 前 8）
cols_ct <- setNames(RColorBrewer::brewer.pal(12,"Paired")[1:6], c("Rod","Cone","AC","BC","HC","MG"))
ct_order   <- c("Rod","Cone","AC","BC","HC","MG")
fate_lv    <- c("DM","DAL","AAM")

# 直接使用脚本 4 的 celltype 立体表（direction×fate×module×celltype 的 pathway/gene 数 + gene list）
src <- read_csv(file.path(dir_d,"2026-09-02_4_fate_module_celltype.csv"), show_col_types=FALSE) %>%
  rename(fate=subclass, n_unique_pathways=n_pathway, n_anchor_genes=n_gene, anchor_gene_list=genes) %>%
  mutate(fate=factor(fate, fate_lv), celltype=factor(celltype, ct_order))
cat("source 行数:", nrow(src), "| direction×fate:\n"); print(table(src$direction, src$fate))

In [ ]:
# ---- 补 pathway_list（source 表需要），从连接主表取 --------------------------
conn <- read_csv(file.path(dir_d,"2026-09-02_2_fate_pathway_connection_all.csv"), show_col_types=FALSE) %>% filter(pass_U1)
mm   <- read_csv(file.path(dir_d,"2026-09-02_3_module_map_combined.csv"), show_col_types=FALSE)
pw_list <- conn %>% left_join(mm, by=c("direction","pathway")) %>%
  group_by(direction, subclass, module, celltype) %>%
  summarise(pathway_list=paste(sort(unique(pathway)), collapse=";"), .groups="drop") %>%
  rename(fate=subclass)

# ---- 主表 1：fate_module_celltype_source.csv -------------------------------
source_tbl <- src %>% left_join(pw_list, by=c("direction","fate","module","celltype")) %>%
  select(direction, fate, celltype, module, n_unique_pathways, n_anchor_genes, pathway_list, anchor_gene_list) %>%
  arrange(direction, fate, celltype, module)
write_csv(source_tbl, file.path(dir_d,"2026-09-06_1_fate_module_celltype_source.csv"), na="NA")
cat("source 表:", nrow(source_tbl), "行\n")

In [ ]:
writeLines(c("date_str: 2026-09-06_1  (Cell type -> Module pathway-source network)",
  "input: 2026-09-02_4_fate_module_celltype.csv (+ pathway_list from 2026-09-02_2)",
  "unit: direction x fate x celltype x module",
  "link width: two versions -- n_unique_pathways & n_anchor_genes",
  "celltype order: Rod->Cone->AC->BC->HC->MG; module order shared across directions (by total anchor genes)",
  "both directions same pipeline; NO cross-celltype NES averaging",
  "interpretation: cross-celltype presence = module-level convergence only (not shared genes/pathways)",
  "colors: /project/work/color.ipynb cols.celltype (Paired)",
  "", capture.output(sessionInfo())),
  file.path(dir_d,"2026-09-06_1_parameters_sessionInfo.txt"))
cat("完成\n")